# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamarBabar02/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

### Chosen method: Logistic Regression

My lane is Content Opportunity Scoring / Refresh ranking, so the goal is to assign scores to pages and rank them by their likelihood of being a useful refresh opportunity.

I chose Logistic Regression as the first learned model because it is simple, interpretable, and produces probability scores that can be used to rank items.

This is appropriate as a first model because it provides a transparent learned-model comparison against the Week-4 rule-based baseline. I will evaluate both approaches using the same evaluation rows and the same ranking metrics.

I will only consider a more complex model if the results show that additional complexity provides a meaningful improvement.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I will use a grouped train/test split by client_hash_id.

The starter dataset contains content items from multiple clients. The client identifier is used only for grouping and splitting, not as a model feature.

Grouping by client is an honest split because it prevents content from the same client from appearing in both the training and test sets. This reduces the risk that the model learns client-specific patterns and then receives an artificially easy test set.

The Week-4 baseline did not use a train/test split; it was a March 2026 rule-based ranking. Therefore, in Week 5 I will evaluate that baseline rule on the same held-out test rows used for the Logistic Regression model. This makes the comparison fair without claiming that Week 4 already had a train/test split.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

The target is whether a content item becomes a CTR improvement opportunity.

To avoid using future information, the target is constructed from the observed March 2026 data available in this modeling dataset. The model is trained only on the training clients and evaluated on unseen clients.

The model uses search-performance features that are available at scoring time. Leakage-prone fields such as trend_pct and trend_direction are not used.

The model probability is used as the learned ranking score.

The Week-4 rule-based baseline is also applied to exactly the same test rows. Both methods are evaluated using Precision@20 and Precision@50.

In [1]:
import os
import numpy as np
import pandas as pd
import duckdb

from google.colab import userdata

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

RANDOM_STATE = 42

print("Imports successful.")

Imports successful.


In [2]:
!pip -q install duckdb huggingface_hub

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face connection configured successfully.")

Hugging Face connection configured successfully.


In [3]:
rel = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
"""

query = f"""
WITH base AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_sessions,
        ga4_engaged_sessions,
        ga4_total_engagement_sec,
        sessions_organic,
        sessions_direct,
        sessions_referral,
        sessions_social,
        sessions_paid,
        sessions_ai,
        scroll_events
    FROM {rel}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-30'
),

current_data AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_sessions,
        ga4_engaged_sessions,
        ga4_total_engagement_sec,
        sessions_organic,
        sessions_direct,
        sessions_referral,
        sessions_social,
        sessions_paid,
        sessions_ai,
        scroll_events,

        CASE
            WHEN gsc_impressions > 0
            THEN gsc_clicks * 1.0 / gsc_impressions
            ELSE NULL
        END AS ctr

    FROM base
),

next_day AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        CASE
            WHEN gsc_impressions > 0
            THEN gsc_clicks * 1.0 / gsc_impressions
            ELSE NULL
        END AS next_ctr
    FROM base
)

SELECT
    c.*,
    n.next_ctr,

    CASE
        WHEN c.ctr IS NOT NULL
             AND n.next_ctr IS NOT NULL
             AND n.next_ctr > c.ctr
        THEN 1
        ELSE 0
    END AS target

FROM current_data c

LEFT JOIN next_day n
    ON c.client_hash_id = n.client_hash_id
    AND c.content_hash_id = n.content_hash_id
    AND n.report_date = c.report_date + INTERVAL 1 DAY

WHERE c.gsc_impressions > 0
  AND c.gsc_avg_position > 0
  AND n.next_ctr IS NOT NULL
  AND c.gsc_impressions >= 5
"""

print("Running DuckDB query...")

training_df = con.sql(query).df()

print("Training dataset shape:", training_df.shape)
training_df.head()

Running DuckDB query...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training dataset shape: (2409018, 20)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,scroll_events,ctr,next_ctr,target
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.000000,0.000000,0
1,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.008000,0.002611,0
2,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.000000,0.000000,0
3,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.000000,0.000000,0
4,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.004184,0.000000,0


In [4]:
print("Target distribution:")
print(training_df["target"].value_counts())

print("\nTarget proportion:")
print(training_df["target"].value_counts(normalize=True))

Target distribution:
target
0    2123102
1     285916
Name: count, dtype: int64

Target proportion:
target
0    0.881314
1    0.118686
Name: proportion, dtype: float64


In [5]:
MAX_ROWS = 300000

if len(training_df) > MAX_ROWS:
    training_df = (
        training_df
        .sample(n=MAX_ROWS, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )

print("Final ML dataset shape:", training_df.shape)

Final ML dataset shape: (300000, 20)


In [6]:
training_df["ctr_bucket"] = pd.cut(
    training_df["ctr"],
    bins=[-np.inf, 0.02, 0.05, np.inf],
    labels=["Low", "Medium", "High"]
)

training_df["position_bucket"] = pd.cut(
    training_df["gsc_avg_position"],
    bins=[-np.inf, 3, 10, np.inf],
    labels=["Strong (1-3)", "Moderate (4-10)", "Weak (>10)"]
)

valid_position = training_df["gsc_avg_position"].where(
    training_df["gsc_avg_position"] > 0
)

training_df["baseline_score"] = (
    (1 - training_df["ctr"].clip(0, 1)) * 0.6
    + (1 / (1 + valid_position)) * 0.4
)

print("Baseline score created.")

Baseline score created.


In [7]:
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]

X = training_df[feature_cols].copy()
y = training_df["target"].astype(int)
groups = training_df["client_hash_id"]

print("Number of features:", len(feature_cols))
print("Features:")
print(feature_cols)

Number of features: 15
Features:
['gsc_impressions', 'gsc_clicks', 'ctr', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'scroll_events']


In [8]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_rows = training_df.iloc[train_idx].copy()
test_rows = training_df.iloc[test_idx].copy()

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print(
    "Unique train clients:",
    train_rows["client_hash_id"].nunique()
)

print(
    "Unique test clients:",
    test_rows["client_hash_id"].nunique()
)

overlap = set(train_rows["client_hash_id"]).intersection(
    set(test_rows["client_hash_id"])
)

print("Client overlap:", len(overlap))

Training rows: 282214
Test rows: 17786
Unique train clients: 35
Unique test clients: 9
Client overlap: 0


In [9]:
logistic_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    )
])

logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [10]:
model_probability = logistic_model.predict_proba(X_test)[:, 1]

model_prediction = (
    model_probability >= 0.5
).astype(int)

print("Predictions generated.")

Predictions generated.


In [11]:
accuracy = accuracy_score(
    y_test,
    model_prediction
)

precision = precision_score(
    y_test,
    model_prediction,
    zero_division=0
)

recall = recall_score(
    y_test,
    model_prediction,
    zero_division=0
)

f1 = f1_score(
    y_test,
    model_prediction,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    model_probability
)

print("Logistic Regression Metrics")
print("---------------------------")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("ROC AUC  :", round(roc_auc, 4))

Logistic Regression Metrics
---------------------------
Accuracy : 0.8701
Precision: 0.2835
Recall   : 0.3329
F1 Score : 0.3062
ROC AUC  : 0.7471


In [12]:
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_k_idx = np.argsort(scores)[::-1][:k]

    return y_true[top_k_idx].mean()

In [13]:
test_baseline_scores = test_rows["baseline_score"].to_numpy()

comparison_rows = []

for k in [20, 50, 100]:

    baseline_p = precision_at_k(
        y_test.to_numpy(),
        test_baseline_scores,
        k
    )

    model_p = precision_at_k(
        y_test.to_numpy(),
        model_probability,
        k
    )

    comparison_rows.append({
        "method": "Week-4 Baseline",
        "K": k,
        "precision_at_k": baseline_p
    })

    comparison_rows.append({
        "method": "Logistic Regression",
        "K": k,
        "precision_at_k": model_p
    })

comparison_table = pd.DataFrame(comparison_rows)

display(comparison_table)

,method,K,precision_at_k
0,Week-4 Baseline,20,0.15
1,Logistic Regression,20,0.60
2,Week-4 Baseline,50,0.08
3,Logistic Regression,50,0.60
4,Week-4 Baseline,100,0.06
5,Logistic Regression,100,0.51


In [14]:
base_rate = y_test.mean()

print("Test-set positive base rate:", round(base_rate, 4))

comparison_table["base_rate"] = base_rate

display(comparison_table)

Test-set positive base rate: 0.0861


,method,K,precision_at_k,base_rate
0,Week-4 Baseline,20,0.15,0.086135
1,Logistic Regression,20,0.60,0.086135
2,Week-4 Baseline,50,0.08,0.086135
3,Logistic Regression,50,0.60,0.086135
4,Week-4 Baseline,100,0.06,0.086135
5,Logistic Regression,100,0.51,0.086135


In [15]:
comparison_pivot = comparison_table.pivot(
    index="K",
    columns="method",
    values="precision_at_k"
).reset_index()

comparison_pivot["model_lift"] = (
    comparison_pivot["Logistic Regression"]
    - comparison_pivot["Week-4 Baseline"]
)

display(comparison_pivot)

method,K,Logistic Regression,Week-4 Baseline,model_lift
0,20,0.60,0.15,0.45
1,50,0.60,0.08,0.52
2,100,0.51,0.06,0.45


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The model is evaluated on clients that were not present in training.

For ranking, Precision@20 and Precision@50 are more useful than accuracy because the practical question is which pages should be reviewed first.

I will inspect false positives and false negatives to understand where the model makes mistakes. I will also inspect model coefficients to understand which observed features influence the ranking.

The interpretation is directional: feature importance shows what the model used, not proof that a feature causes the outcome.

In [16]:
test_results = test_rows[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "next_ctr",
        "target",
        "baseline_score"
    ]
].copy()

test_results["model_probability"] = model_probability
test_results["model_prediction"] = model_prediction

test_results["error_type"] = np.select(
    [
        (test_results["target"] == 1) &
        (test_results["model_prediction"] == 0),

        (test_results["target"] == 0) &
        (test_results["model_prediction"] == 1)
    ],
    [
        "False Negative",
        "False Positive"
    ],
    default="Correct"
)

display(
    test_results["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="count")
)

,error_type,count
0,Correct,15475
1,False Positive,1289
2,False Negative,1022


In [17]:
false_positives = (
    test_results[
        test_results["error_type"] == "False Positive"
    ]
    .sort_values(
        "model_probability",
        ascending=False
    )
)

print("Top False Positives")
display(
    false_positives.head(10)
)

Top False Positives


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,next_ctr,target,baseline_score,model_probability,model_prediction,error_type
148716,2026-03-03,client_0fa64a184f18a4a0,content_fe8328ab251759cc,2643,0,0.000000,9.032160,0.000000,0,0.639872,0.999987,1,False Positive
256566,2026-03-29,client_3f0ce4d44fe94f3d,content_0b628d4de0ca17cb,1610,2,0.001242,5.250932,0.000663,0,0.663245,0.998394,1,False Positive
20345,2026-03-29,client_0fa64a184f18a4a0,content_1478ceb98c064458,2800,39,0.013929,1.821071,0.011864,0,0.733433,0.997802,1,False Positive
3430,2026-03-16,client_0fa64a184f18a4a0,content_bbc8345bb63436e1,1447,21,0.014513,2.255701,0.007293,0,0.714154,0.993276,1,False Positive
234455,2026-03-21,client_3f0ce4d44fe94f3d,content_277e38945834136d,936,0,0.000000,3.896368,0.000000,0,0.681693,0.978985,1,False Positive
11994,2026-03-28,client_3f0ce4d44fe94f3d,content_277e38945834136d,926,2,0.002160,3.924406,0.001406,0,0.679932,0.968745,1,False Positive
273733,2026-03-28,client_3f0ce4d44fe94f3d,content_453884ed468172d5,845,1,0.001183,6.080473,0.000000,0,0.655783,0.962966,1,False Positive
116658,2026-03-27,client_0fa64a184f18a4a0,content_809ab284e58267ee,799,7,0.008761,4.381727,0.007722,0,0.669069,0.952386,1,False Positive
281913,2026-03-25,client_3f0ce4d44fe94f3d,content_1bc14f25ea5989e0,705,0,0.000000,7.137589,0.000000,0,0.649155,0.938721,1,False Positive
32715,2026-03-26,client_3f0ce4d44fe94f3d,content_487dfed1dc8dc39a,743,2,0.002692,4.783311,0.001328,0,0.667549,0.930146,1,False Positive


In [18]:
false_negatives = (
    test_results[
        test_results["error_type"] == "False Negative"
    ]
    .sort_values(
        "model_probability",
        ascending=True
    )
)

print("Top False Negatives")
display(
    false_negatives.head(10)
)

Top False Negatives


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,next_ctr,target,baseline_score,model_probability,model_prediction,error_type
103897,2026-03-22,client_f623b01661d4bfe4,content_b8eb55e9605771a2,8,0,0.0,86.125000,0.066667,1,0.604591,0.038824,0,False Negative
178344,2026-03-09,client_9958f0a7ae1df715,content_52d6e0ecebc50434,18,0,0.0,75.888889,0.062500,1,0.605202,0.057246,0,False Negative
38018,2026-03-18,client_f623b01661d4bfe4,content_9abace70356e4fe5,30,0,0.0,77.100000,0.025641,1,0.605122,0.057815,0,False Negative
231295,2026-03-27,client_f623b01661d4bfe4,content_969352f4e556bad4,30,0,0.0,74.533333,0.040000,1,0.605296,0.062979,0,False Negative
181345,2026-03-14,client_2094c6eb080311d5,content_60986b5d4339d7ed,15,0,0.0,72.333333,0.100000,1,0.605455,0.063645,0,False Negative
240374,2026-03-21,client_2094c6eb080311d5,content_0e14f4cf040213af,13,0,0.0,67.000000,0.200000,1,0.605882,0.075273,0,False Negative
43826,2026-03-24,client_f623b01661d4bfe4,content_0ef92dd8ea773a64,74,0,0.0,72.932432,0.062500,1,0.605410,0.079648,0,False Negative
32925,2026-03-15,client_9958f0a7ae1df715,content_fb61c888f78b143f,17,0,0.0,56.117647,0.031250,1,0.607003,0.098700,0,False Negative
238604,2026-03-20,client_9958f0a7ae1df715,content_a6c83856ae9ea54b,5,0,0.0,57.200000,0.055556,1,0.606873,0.100078,0,False Negative
195795,2026-03-22,client_f623b01661d4bfe4,content_81debce7b0565448,23,0,0.0,57.913043,0.035714,1,0.606790,0.105130,0,False Negative


In [19]:
wrong_cases = test_results[
    test_results["error_type"] != "Correct"
].copy()

wrong_cases = wrong_cases.sort_values(
    "model_probability",
    ascending=False
).head(3)

display(
    wrong_cases[
        [
            "report_date",
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "gsc_avg_position",
            "next_ctr",
            "target",
            "model_probability",
            "error_type"
        ]
    ]
)

,report_date,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,next_ctr,target,model_probability,error_type
148716,2026-03-03,content_fe8328ab251759cc,2643,0,0.000000,9.032160,0.000000,0,0.999987,False Positive
256566,2026-03-29,content_0b628d4de0ca17cb,1610,2,0.001242,5.250932,0.000663,0,0.998394,False Positive
20345,2026-03-29,content_1478ceb98c064458,2800,39,0.013929,1.821071,0.011864,0,0.997802,False Positive


In [20]:
model = logistic_model.named_steps["model"]

coefficients = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model.coef_[0]
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Top model features:")
display(coefficients.head(10))

Top model features:


,feature,coefficient,absolute_coefficient
0,gsc_impressions,1.274347,1.274347
3,gsc_avg_position,-0.559472,0.559472
1,gsc_clicks,-0.384849,0.384849
8,sessions_organic,0.165237,0.165237
4,ga4_pageviews,0.059428,0.059428
6,ga4_engaged_sessions,0.042660,0.042660
5,ga4_sessions,0.036599,0.036599
14,scroll_events,-0.033654,0.033654
9,sessions_direct,0.030574,0.030574
10,sessions_referral,0.020984,0.020984


In [21]:
top_3_features = coefficients.head(3)

print("Top 3 features:")
display(
    top_3_features[
        ["feature", "coefficient"]
    ]
)

Top 3 features:


,feature,coefficient
0,gsc_impressions,1.274347
3,gsc_avg_position,-0.559472
1,gsc_clicks,-0.384849


In [22]:
error_summary = (
    test_results[
        test_results["error_type"] != "Correct"
    ]
    .groupby("error_type")
    .agg(
        cases=("content_hash_id", "count"),
        avg_impressions=("gsc_impressions", "mean"),
        avg_ctr=("ctr", "mean"),
        avg_position=("gsc_avg_position", "mean"),
        avg_probability=("model_probability", "mean")
    )
    .reset_index()
)

display(error_summary)

,error_type,cases,avg_impressions,avg_ctr,avg_position,avg_probability
0,False Negative,1022,37.554795,0.003560,9.633244,0.411457
1,False Positive,1289,154.976726,0.006984,4.432873,0.578781


In [23]:
print("Random state:", RANDOM_STATE)
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Number of features:", len(feature_cols))

print(
    "Client overlap between train/test:",
    len(
        set(train_rows["client_hash_id"])
        .intersection(
            set(test_rows["client_hash_id"])
        )
    )
)

print("Reproducibility checks complete.")

Random state: 42
Train rows: 282214
Test rows: 17786
Number of features: 15
Client overlap between train/test: 0
Reproducibility checks complete.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.